In [40]:
import pandas as pd
import json

with open('mulCitingMeta.json') as data_file:    
    data = json.load(data_file)

In [41]:
from bson.json_util import dumps
import sys
sys.path.append("/home/pop533611/citation-recommendation/utils")
import mongoConnect as mc

In [3]:
from bson.objectid import ObjectId

mulCitingMetaData = []
for d in data:
    # print(d["_id"]["$oid"])
#     "_id": {
#     "$oid": "647791243bc8331c176e8c24"
#   },
    records = mc.get_records("citedData",{"_id": ObjectId(d["citedId"]["$oid"])})
    for rec in records:
        d["ref"] = rec["ref"]
        mulCitingMetaData.append(d)

print(len(mulCitingMetaData))        
with open('mulCitingMeta.json','w') as data_file:    
    json_object = json.dump(mulCitingMetaData, data_file)


838


In [50]:
from bson.objectid import ObjectId

mulCitedMetaData = []

for d in data:
    pipeline = [
    {
        "$match": {"_id": ObjectId(d["citedId"]["$oid"])}
    },
    {
        "$lookup": {
            "from": "papers",
            "localField": "paperId",
            "foreignField": "_id",
            "as": "metaData"
        }
    }
    ]
    res = mc.agg_query("citedData",pipeline)
   
    for r in res:
        metainfo = {
            "title" : r["metaData"][0]["title"],
            "year" : r["metaData"][0]["year"],
            "citationCount": r["metaData"][0]["citationCount"],
            "venue": r["metaData"][0]["venue"],
            "author":r["metaData"][0]["author"]
        }
       
        d["metaData"] = metainfo
        
    mulCitedMetaData.append(d)
       
with open('mulCitedMetaData.json','w') as data_file:    
    json_object = json.dump(mulCitedMetaData, data_file)

In [51]:
len(mulCitedMetaData)

838